# 🚀 机械臂扭矩曲线预测 - 优化版训练

本笔记本包含完整的训练流程，使用优化的模型配置。

## 主要改进
- ✅ 使用 Transformer 模型（注意力机制）
- ✅ 增加模型容量（hidden_dim=256, num_layers=4）
- ✅ 降低学习率提高稳定性
- ✅ 增加训练轮次和早停耐心值
- ✅ 在预测图上显示 R² 指标

## 预期效果
- R² > 0.96
- MAPE < 50%

## 1. 环境设置

In [ ]:
# 如果在 Colab 中运行，先克隆仓库
import os

if not os.path.exists('Super-Strawberry'):
    # 方法1: 使用 Personal Access Token
    # !git clone https://<YOUR_TOKEN>@github.com/your-username/Super-Strawberry.git
    
    # 方法2: 公开仓库直接克隆
    # !git clone https://github.com/your-username/Super-Strawberry.git
    
    print("⚠️ 请手动设置仓库 URL 并克隆")
else:
    print("✅ 仓库已存在")

# 切换到项目目录
%cd Super-Strawberry

In [ ]:
# 安装依赖
!pip install -q torch numpy pandas matplotlib seaborn scikit-learn tqdm

## 2. 导入库和模块

In [ ]:
import sys
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 添加项目根目录到路径
project_root = os.path.abspath('.')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 导入自定义模块
from src.data_loader import load_data, get_sample_data_info
from src.model import get_model
from src.train import Trainer, predict_batch, calculate_metrics
from src.evaluate import evaluate_model, plot_training_history, plot_predictions

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ CUDA device: {torch.cuda.get_device_name(0)}")

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

## 3. 配置参数

In [ ]:
# ==================== 数据配置 ====================
DATA_DIR = './data'
SIGNAL_TYPE = 'signal_1'          # 预测的信号类型
INPUT_LENGTH = 100                # 输入序列长度
OUTPUT_LENGTH = 400               # 输出序列长度
USE_ALL_FEATURES = True           # 使用所有特征

# ==================== 模型配置（优化版）====================
MODEL_TYPE = 'transformer'        # 模型类型: 'lstm', 'gru', 'transformer', 'simple_lstm'
INPUT_DIM = 4                     # ⚠️ 输入特征维度（Time, signal_0, signal_1, signal_2）
HIDDEN_DIM = 256                  # 🔥 隐藏层维度（从 128 增加到 256）
NUM_LAYERS = 4                    # 🔥 网络层数（从 2-3 增加到 4）
DROPOUT = 0.3                     # Dropout 比例

# ==================== 训练配置（优化版）====================
BATCH_SIZE = 64                   # 🔥 批次大小（从 32 增加到 64）
LEARNING_RATE = 0.0005            # 🔥 学习率（从 0.001 降低到 0.0005）
EPOCHS = 150                      # 🔥 训练轮数（从 100 增加到 150）
EARLY_STOPPING_PATIENCE = 20      # 🔥 早停耐心值（从 10-15 增加到 20）

# ==================== 设备配置 ====================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ==================== 路径配置 ====================
MODEL_SAVE_DIR = './models_optimized'
RESULTS_DIR = './results_optimized'

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# 打印配置
print("=" * 60)
print("训练配置")
print("=" * 60)
print(f"数据目录:         {DATA_DIR}")
print(f"预测信号:         {SIGNAL_TYPE}")
print(f"输入长度:         {INPUT_LENGTH}")
print(f"输出长度:         {OUTPUT_LENGTH}")
print(f"模型类型:         {MODEL_TYPE}")
print(f"输入维度:         {INPUT_DIM}")
print(f"隐藏维度:         {HIDDEN_DIM}")
print(f"网络层数:         {NUM_LAYERS}")
print(f"批次大小:         {BATCH_SIZE}")
print(f"学习率:           {LEARNING_RATE}")
print(f"训练轮数:         {EPOCHS}")
print(f"计算设备:         {DEVICE}")
print("=" * 60)

## 4. 数据准备

### 4.1 生成样本数据（如果需要）

In [ ]:
# 检查数据目录
import glob

csv_files = glob.glob(os.path.join(DATA_DIR, '*.csv'))

if len(csv_files) == 0:
    print("❌ 未找到数据文件！")
    print("⚠️ 请运行上面的单元格从 Google Drive 下载数据")
else:
    print(f"✅ 找到 {len(csv_files)} 个数据文件")

In [ ]:
# 检查数据目录
import glob

csv_files = glob.glob(os.path.join(DATA_DIR, '*_open.csv'))

if len(csv_files) == 0:
    print("⚠️ 未找到数据文件，正在生成样本数据...")
    !python generate_sample_data.py
    csv_files = glob.glob(os.path.join(DATA_DIR, '*_open.csv'))
    print(f"✅ 生成了 {len(csv_files)} 个样本数据文件")
else:
    print(f"✅ 找到 {len(csv_files)} 个数据文件")

### 4.2 查看数据信息

In [ ]:
# 查看第一个数据文件
if len(csv_files) > 0:
    sample_df = get_sample_data_info(csv_files[0])
    
    # 可视化信号
    plt.figure(figsize=(15, 4))
    plt.plot(sample_df['Time(s)'], sample_df['signal_0'], label='signal_0', alpha=0.7)
    plt.plot(sample_df['Time(s)'], sample_df['signal_1'], label='signal_1', alpha=0.7)
    plt.plot(sample_df['Time(s)'], sample_df['signal_2'], label='signal_2', alpha=0.7)
    plt.xlabel('Time (s)')
    plt.ylabel('Signal Value')
    plt.title('Sample Signal Visualization')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print("加载数据...")

train_loader, test_loader = load_data(
    data_dir=DATA_DIR,
    pattern='*.csv',  # 匹配所有 CSV 文件
    train_split=0.8,
    input_length=INPUT_LENGTH,
    output_length=OUTPUT_LENGTH,
    signal_type=SIGNAL_TYPE,
    use_all_features=USE_ALL_FEATURES,
    batch_size=BATCH_SIZE
)

# 检查数据形状
for inputs, targets in train_loader:
    print(f"\n✅ 数据加载成功！")
    print(f"输入形状: {inputs.shape}  # [batch_size, input_length, features]")
    print(f"输出形状: {targets.shape}  # [batch_size, output_length]")
    print(f"特征数量: {inputs.shape[2]}")
    break

In [ ]:
print("加载数据...")

train_loader, test_loader = load_data(
    data_dir=DATA_DIR,
    pattern='*_open.csv',
    train_split=0.8,
    input_length=INPUT_LENGTH,
    output_length=OUTPUT_LENGTH,
    signal_type=SIGNAL_TYPE,
    use_all_features=USE_ALL_FEATURES,
    batch_size=BATCH_SIZE
)

# 检查数据形状
for inputs, targets in train_loader:
    print(f"\n✅ 数据加载成功！")
    print(f"输入形状: {inputs.shape}  # [batch_size, input_length, features]")
    print(f"输出形状: {targets.shape}  # [batch_size, output_length]")
    print(f"特征数量: {inputs.shape[2]}")
    break

## 5. 创建模型

In [ ]:
print("创建模型...")

model = get_model(
    model_type=MODEL_TYPE,
    input_dim=INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    output_length=OUTPUT_LENGTH,
    dropout=DROPOUT
)

# 计算模型参数数量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✅ 模型创建成功！")
print(f"模型类型:     {MODEL_TYPE}")
print(f"总参数数:     {total_params:,}")
print(f"可训练参数:   {trainable_params:,}")
print(f"\n模型结构:")
print(model)

## 6. 训练模型

In [ ]:
print("\n创建训练器...")

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    device=DEVICE,
    learning_rate=LEARNING_RATE,
    save_dir=MODEL_SAVE_DIR
)

print("\n" + "=" * 60)
print("开始训练...")
print("=" * 60)

history = trainer.train(
    epochs=EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE
)

print("\n✅ 训练完成！")

## 7. 训练历史可视化

In [ ]:
# 绘制训练历史
plot_training_history(
    history,
    save_path=os.path.join(RESULTS_DIR, 'training_history.png')
)

## 8. 模型评估

In [ ]:
print("\n加载最佳模型...")

# 加载最佳模型
trainer.load_checkpoint('best_model.pth')

print("\n" + "=" * 60)
print("评估模型...")
print("=" * 60)

# 完整评估
metrics, predictions, targets, inputs = evaluate_model(
    model=trainer.model,
    data_loader=test_loader,
    device=DEVICE,
    save_dir=RESULTS_DIR
)

## 9. 结果分析

In [ ]:
# 打印详细指标
print("\n" + "=" * 60)
print("📊 最终评估指标")
print("=" * 60)

for key, value in metrics.items():
    if key == 'R2':
        print(f"{key:10s}: {value:.6f}  ({'✅ 优秀' if value > 0.96 else '⚠️ 需要改进' if value > 0.90 else '❌ 较差'})")
    elif key == 'MAPE':
        print(f"{key:10s}: {value:.2f}%    ({'✅ 优秀' if value < 30 else '⚠️ 需要改进' if value < 50 else '❌ 较差'})")
    else:
        print(f"{key:10s}: {value:.6f}")

print("=" * 60)

# 与基线对比（假设之前的结果）
print("\n📈 与基线模型对比:")
baseline_r2 = 0.949
baseline_mape = 63.5

r2_improvement = (metrics['R2'] - baseline_r2) / baseline_r2 * 100
mape_improvement = (baseline_mape - metrics['MAPE']) / baseline_mape * 100

print(f"R² 提升:   {r2_improvement:+.2f}% ({baseline_r2:.4f} → {metrics['R2']:.4f})")
print(f"MAPE 降低: {mape_improvement:+.2f}% ({baseline_mape:.2f}% → {metrics['MAPE']:.2f}%)")

## 10. 单样本预测演示

In [ ]:
# 随机选择一个测试样本
sample_idx = np.random.randint(0, len(predictions))

# 绘制单个预测
fig, ax = plt.subplots(1, 1, figsize=(15, 5))

input_len = inputs.shape[1]
output_len = predictions.shape[1]

# 时间轴
input_time = np.arange(0, input_len)
output_time = np.arange(input_len, input_len + output_len)

# 绘制输入序列
input_signal = inputs[sample_idx, :, -2] if inputs.shape[2] >= 4 else inputs[sample_idx, :, 0]
ax.plot(input_time, input_signal, 'b-', label='Input Sequence', linewidth=2, alpha=0.7)

# 绘制真实值和预测值
ax.plot(output_time, targets[sample_idx], 'g-', label='Ground Truth', linewidth=2, alpha=0.7)
ax.plot(output_time, predictions[sample_idx], 'r--', label='Prediction', linewidth=2, alpha=0.7)

# 添加分界线
ax.axvline(x=input_len, color='gray', linestyle='--', alpha=0.5, label='Prediction Start')

# 计算误差
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
mse = mean_squared_error(targets[sample_idx], predictions[sample_idx])
mae = mean_absolute_error(targets[sample_idx], predictions[sample_idx])
r2 = r2_score(targets[sample_idx], predictions[sample_idx])

ax.set_xlabel('Time Step', fontsize=12)
ax.set_ylabel('Signal Value', fontsize=12)
ax.set_title(f'Sample {sample_idx + 1} - MSE: {mse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}', 
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'single_prediction_demo.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✅ 单样本预测演示图已保存到: {os.path.join(RESULTS_DIR, 'single_prediction_demo.png')}")

## 11. 保存结果总结

In [ ]:
# 创建结果总结
summary = {
    'model_type': MODEL_TYPE,
    'input_dim': INPUT_DIM,
    'hidden_dim': HIDDEN_DIM,
    'num_layers': NUM_LAYERS,
    'dropout': DROPOUT,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'epochs_trained': len(history['train_loss']),
    'total_params': total_params,
    'device': DEVICE,
    **metrics
}

# 保存为 JSON
import json
with open(os.path.join(RESULTS_DIR, 'training_summary.json'), 'w') as f:
    json.dump(summary, f, indent=4)

print("\n" + "=" * 60)
print("✅ 训练完成！结果已保存")
print("=" * 60)
print(f"模型保存在:   {MODEL_SAVE_DIR}")
print(f"结果保存在:   {RESULTS_DIR}")
print(f"\n生成的文件:")
print(f"  - {MODEL_SAVE_DIR}/best_model.pth")
print(f"  - {MODEL_SAVE_DIR}/final_model.pth")
print(f"  - {MODEL_SAVE_DIR}/training_history.json")
print(f"  - {RESULTS_DIR}/training_history.png")
print(f"  - {RESULTS_DIR}/predictions.png")
print(f"  - {RESULTS_DIR}/error_distribution.png")
print(f"  - {RESULTS_DIR}/metrics.csv")
print(f"  - {RESULTS_DIR}/training_summary.json")
print("=" * 60)

## 12. 进一步优化建议

如果效果还不够理想，可以尝试：

### 方案 1: 尝试不同的模型
```python
# 回到第5节，修改 MODEL_TYPE
MODEL_TYPE = 'lstm'  # 或 'gru'
```

### 方案 2: 增加模型容量
```python
HIDDEN_DIM = 512  # 进一步增加
NUM_LAYERS = 6
```

### 方案 3: 调整学习率
```python
LEARNING_RATE = 0.0003  # 更低的学习率
EPOCHS = 200            # 更多轮次
```

### 方案 4: 集成多个模型
查看 `OPTIMIZATION_GUIDE.md` 中的集成学习方案

---

## 📚 相关文档
- [README.md](README.md) - 项目概述
- [DATA_FORMAT.md](DATA_FORMAT.md) - 数据格式说明
- [OPTIMIZATION_GUIDE.md](OPTIMIZATION_GUIDE.md) - 详细优化指南